# LLM-Judge Validation Notebook

Purpose: before trusting the LLM-judge to score the full SpeechAgentEval test suite,
validate it against your own manual labels on a subset (target: 50-75 samples spanning
all taxonomy categories).

Workflow:
1. Manually label a sample of pipeline runs in `manual_labels_template.csv`
   (fill in `manual_b_label` and `manual_c_label` columns yourself, using the
   definitions in `judge_prompt.py` / `TAXONOMY.md`).
2. Run the LLM-judge on the same samples.
3. Compare agreement (accuracy + Cohen's kappa per axis).
4. If agreement is weak on any category, refine the judge prompt and re-run
   this notebook before trusting it at scale.

Do not skip step 4 if kappa is low — report the refinement in your writeup,
it's evidence of rigor, not a flaw.

In [ ]:
import os
import json
import time
import pandas as pd
from sklearn.metrics import cohen_kappa_score, accuracy_score, confusion_matrix
import anthropic

from judge_prompt import SYSTEM_PROMPT, build_user_prompt

# Requires ANTHROPIC_API_KEY set in environment
client = anthropic.Anthropic()
MODEL = "claude-sonnet-4-6"  # swap for whichever model you're using as judge


## 1. Load manually labeled samples

Fill out `manual_labels_template.csv` first — every row needs `manual_b_label` and `manual_c_label` filled in.

In [ ]:
df = pd.read_csv("manual_labels_template.csv")
df = df.dropna(subset=["manual_b_label", "manual_c_label"])
print(f"Loaded {len(df)} manually labeled samples")
df.head()


## 2. Run the LLM-judge on the same samples

In [ ]:
def call_judge(ground_truth, asr_transcript, agent_response, max_retries=3):
    user_prompt = build_user_prompt(ground_truth, asr_transcript, agent_response)
    for attempt in range(max_retries):
        try:
            resp = client.messages.create(
                model=MODEL,
                max_tokens=200,
                system=SYSTEM_PROMPT,
                messages=[{"role": "user", "content": user_prompt}],
            )
            text = resp.content[0].text.strip()
            text = text.replace("```json", "").replace("```", "").strip()
            return json.loads(text)
        except (json.JSONDecodeError, Exception) as e:
            if attempt == max_retries - 1:
                print(f"Failed after {max_retries} attempts: {e}")
                return {"b_label": "ERROR", "c_label": "ERROR", "confidence": "low", "justification": str(e)}
            time.sleep(2)

judge_results = []
for _, row in df.iterrows():
    result = call_judge(row["ground_truth"], row["asr_transcript"], row["agent_response"])
    result["sample_id"] = row["sample_id"]
    judge_results.append(result)
    time.sleep(0.5)  # gentle rate limiting

judge_df = pd.DataFrame(judge_results)
judge_df.head()


## 3. Merge and compute agreement

In [ ]:
merged = df.merge(judge_df, on="sample_id", suffixes=("", "_judge"))

# Filter out errored judge calls before scoring
valid = merged[(merged["b_label"] != "ERROR") & (merged["c_label"] != "ERROR")]
print(f"{len(valid)}/{len(merged)} judge calls succeeded")

b_acc = accuracy_score(valid["manual_b_label"], valid["b_label"])
c_acc = accuracy_score(valid["manual_c_label"], valid["c_label"])

b_kappa = cohen_kappa_score(valid["manual_b_label"], valid["b_label"])
c_kappa = cohen_kappa_score(valid["manual_c_label"], valid["c_label"])

print(f"Axis B (propagation) — accuracy: {b_acc:.2%}, Cohen's kappa: {b_kappa:.3f}")
print(f"Axis C (agent behavior) — accuracy: {c_acc:.2%}, Cohen's kappa: {c_kappa:.3f}")


### Interpreting kappa

- **> 0.80**: near-perfect agreement, trust the judge
- **0.60 - 0.80**: substantial agreement, usable but inspect disagreements
- **0.40 - 0.60**: moderate — refine the prompt (add few-shot examples for the
  confused categories) and re-validate before using at scale
- **< 0.40**: weak — do not use the judge as-is; the taxonomy boundary itself
  may need to be clarified, not just the prompt wording

## 4. Inspect disagreements

Look at exactly where manual and judge labels diverge — this tells you whether to fix the prompt, add examples, or the taxonomy boundary itself is ambiguous.

In [ ]:
disagreements_b = valid[valid["manual_b_label"] != valid["b_label"]]
disagreements_c = valid[valid["manual_c_label"] != valid["c_label"]]

print(f"Axis B disagreements: {len(disagreements_b)}")
print(f"Axis C disagreements: {len(disagreements_c)}")

disagreements_b[["sample_id", "ground_truth", "asr_transcript", "agent_response",
                 "manual_b_label", "b_label", "justification"]]


In [ ]:
print("Axis B confusion matrix:")
labels_b = sorted(set(valid["manual_b_label"]) | set(valid["b_label"]))
print(pd.DataFrame(
    confusion_matrix(valid["manual_b_label"], valid["b_label"], labels=labels_b),
    index=labels_b, columns=labels_b
))

print("\nAxis C confusion matrix:")
labels_c = sorted(set(valid["manual_c_label"]) | set(valid["c_label"]))
print(pd.DataFrame(
    confusion_matrix(valid["manual_c_label"], valid["c_label"], labels=labels_c),
    index=labels_c, columns=labels_c
))


## 5. Next steps

- If kappa is acceptable (>0.6) on both axes: save this notebook's output as your
  validation evidence for `REPORT.md`, and move on to running the judge over the
  full test suite.
- If not: edit `judge_prompt.py` (usually: add a few-shot example drawn from an
  actual disagreement above), re-run this notebook, and re-check kappa. Keep a
  record of prompt versions and their kappa scores — this iteration history is
  worth including in your writeup as evidence of a rigorous validation process.